In [1]:
import socket
import json
import pandas as pd
import numpy as np
# # 0   DateTime                                         
#  1   Soil temperture at 2 inch below the soil surface (F)      
#  2   Soil data under the cover right on top of soil surface (F)  
#  3   Air temperature 1.5m (F)                         
#  4   Dew Point                                                
#  5   Wind Chill                                                
#  6   Wind Gust 10m                                               
#  7   Wind Speed 10m                                              
#  8   Wind Direction 10m                                        
#  9   Pressure (mb)                                               
#  10  Pressure (inHg)                                          
#  11  Rainfall                                                   
#  12  Relative Humidity                                         
#  13  Approximate Max                                           
#  14  Solar Radiation                                           
#  15  Snow cover                                                 
#  16  Turfgrass species                                         
#  17  Mowing height                                          
#  18  Location                                            
#  19  Cover or not                                              
#  20  Unnamed: 20                                           





# System Call Log Creation:

In [2]:
import csv
# Write header
#writer.writerow(["Itr_Num", "Layer_Name", "Process_Name", "Log_Prompt"])
global Track_log
Track_log=1
def get_last_track_log(csv_filename):
    try:
        with open(csv_filename, mode="r", newline="") as file:
            reader = csv.reader(file)
            rows = list(reader)
            if rows:  # Ensure the file is not empty
                last_row = rows[-1]  # Get the last row
                return (last_row[-1])  # Extract the last column (Track_log)
            else:
                return 1  # Default value if file is empty
    except FileNotFoundError:
        return 1  # Default if file doesn't exist


def Call_log( Layer_Name, Process_Name, Log_Prompt):
    # Define the CSV file name
    csv_filename = "System_Call_Log.csv"
    # Write data to CSV file
    global Track_log
    Track_log=get_last_track_log(csv_filename)
    itr_num=Track_log
    with open(csv_filename, mode="a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([itr_num, Layer_Name, Process_Name, Log_Prompt,Track_log])

In [3]:


columns=[
    'DateTime','Soil_temperture_at_2_inch_below_the_soil_surface_(F)','Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)',
    'Air_temperature_1.5m_(F)','Dewpoint','Wind_Chill','Wind_Gust_10m','Wind_Speed_10m','Wind_Direction_10m',
    'Pressure_(mb)','Pressure_(inHg)','Rainfall','Relative_Humidity',
    'Approximate_Max','Solar_Radiation','Snow_cover','Turfgrass_species','Mowing_height','Location','Unnamed:_20'
]

In [4]:
def clean_data(value):
    try:
        # Attempt to convert to float, if it fails return NaN
        return float(value.replace('NaN]', '').strip()) if value else np.nan
    except ValueError:
        return np.nan

In [5]:
# Initialize an empty DataFrame globally


df = pd.DataFrame({'DateTime':[ '2/1/2022 16:30']})


**AggregateSensorData** :while edge layer is constantly receiveing sensor data, we prepare them to send to Cloud layer for further  analysis at a regular interval.This process collects the sensor data for a regular interval and then get the mean of the received data grouped by ['Snow_cover', 'Turfgrass_species', 'Mowing_height', 'Location']
**SendSensorDataToCloud**:: handles the data sending to cloud

In [6]:
import threading
import time

# Function to calculate the average every 5 minutes and send it via TCP
def AggregateSensorData(tcp_ip, tcp_port): 
    global df
    while True:
        time.sleep(15)  # give the waiting period
        
        if not df.empty:
            # Group by specific columns and calculate the mean for numeric columns
            print(df)
            grouped_df = df.groupby(['Snow_cover', 'Turfgrass_species', 'Mowing_height', 'Location']).mean().reset_index()
            #df.groupby(['Snow cover', 'Turfgrass species', 'Mowing height', 'Location']).mean()
            print("Average values over the last 5 minutes, grouped by specific columns:")
            print(grouped_df)
            
            # Convert the DataFrame to JSON for transmission
            mean_data_json = grouped_df.to_json()
            Call_log( "Edge Layer", "AggregateSensorData", "Sensor Data has been Aggregated")
            # Send the mean DataFrame to the TCP server
            SendSensorDataToCloud(mean_data_json, tcp_ip, tcp_port) #SendSensorDataToCloud
            
            # Reset the DataFrame for the next cycle
            df = pd.DataFrame(columns=columns)
            print("DataFrame reset for the next cycle.")

# Function to send data via TCP
def SendSensorDataToCloud(data, ip, port):
    # Create a TCP/IP socket
    tcp_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    print("TCP Server" )
    try:
        # Connect to the TCP server
        tcp_socket.connect((ip, port))
        
        # Send the data
        tcp_socket.sendall(data.encode('utf-8'))
        Call_log( "Edge Layer", "SendSensorDataToCloud", "Sensor Data has been sent to cloud")
        print(f"Sent mean DataFrame to TCP server at {ip}:{port}")
    finally:
        tcp_socket.close()


# # Start the average calculation and sending in a separate thread
# # tcp_ip = '127.0.0.1'  # Replace with your TCP server IP
# # tcp_port = 12346      # Replace with your TCP server port
# # Get the IP address of the machine
# hostname = socket.gethostname()
# tcp_ip = socket.gethostbyname(hostname)

# # Choose a port number
# tcp_port = 12346
# threading.Thread(target=calculate_and_send_average, args=(tcp_ip, tcp_port), daemon=True).start()


If Cloud analysis needs to activate some trigger on physical level,it send back to Edge layer. **listen_for_cloud_data** handles the part on receiving data. **SendTriggerToActuator**  then send data to physical layer

In [7]:
# Function to receive data from the cloud via TCP
def listen_for_cloud_data(tcp_ip, tcp_port, physical_ip, physical_port):
    # Create a TCP listener socket for receiving data from the cloud
    tcp_listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    tcp_listener.bind((tcp_ip, tcp_port))
    tcp_listener.listen(1)  # Listen for incoming connections

    print(f"Edge server listening for cloud data on {tcp_ip}:{tcp_port}")
    
    while True:
        conn, addr = tcp_listener.accept()  # Accept a connection from the cloud server
        print(f"Connected to cloud server at {addr}")

        # Receive the data from the cloud
        data_from_cloud = conn.recv(1024).decode('utf-8')  # Adjust buffer size as necessary
        print(f"Data received from cloud: {data_from_cloud}")

        # Process the received data if needed
         # Forward the data to the physical server
        SendTriggerToActuator(data_from_cloud, physical_ip, physical_port)#SendTriggerToActuator
        conn.close()


In [8]:
# Function to forward data to the physical server via TCP
def SendTriggerToActuator(data, physical_ip, physical_port):
    tcp_client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    try:
        tcp_client.connect((physical_ip, physical_port))
        tcp_client.sendall(data.encode('utf-8'))
        print(f"Data forwarded to physical server: {data}")
        Call_log( "Edge Layer", "SendTriggerToActuator", "Trigger actuator has been sent to physical layer")
    except Exception as e:
        print(f"Error sending data to physical server: {e}")
    finally:
        tcp_client.close()

**CollectSensorData()** acts like a UDP server. It stays active to receive sensor data from physical layer.sensor data comes as json 
format. then we load it in a pandas dataframe.

In [9]:

def CollectSensorData(ip, port): #UDP server
    global df  # Declare df as global to modify the global variable
    # Create a UDP socket
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

    # Bind the socket to the address
    server_socket.bind((ip, port))

    print(f"Server listening on {ip}:{port}")
    print("print Df",df)
    while True:
        # Receive data
        data, addr = server_socket.recvfrom(1024)  # Buffer size is 1024 bytes
        message = data.decode('utf-8')
        # Deserialize the data from JSON format
        received_data = json.loads(message)
        print(f"Received data from {addr}: {received_data}")
        #print(f"Received data from {addr}")
        Call_log( "Edge Layer", "CollectSensorData", "Sensor Data has been collected")
        
        #print("parsed_data",parsed_data)
        # Step 3: Parse the received data
        parsed_data=received_data
        # Step 4 : Ensure data matches column count and append to DataFrame
        ##Clean Sensor DATA
        if len(parsed_data) == len(columns):
            temp_df = pd.DataFrame([parsed_data], columns=columns)
            temp_df = temp_df.apply(pd.to_numeric, errors='ignore')
      
        # Step 5: Convert columns to their appropriate data types
     
        temp_df = temp_df.astype({
            'DateTime': 'object',  # or pd.to_datetime() if datetime format
            'Soil_temperture_at_2_inch_below_the_soil_surface_(F)': 'float64',
            'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)': 'float64',
            'Air_temperature_1.5m_(F)': 'float64',
            'Dewpoint': 'float64',
            'Wind_Chill': 'float64',
            'Wind_Gust_10m': 'float64',
            'Wind_Speed_10m': 'float64',
            'Wind_Direction_10m': 'float64',
            'Pressure_(mb)': 'float64',
            'Pressure_(inHg)': 'float64',
            'Rainfall': 'float64',
            'Relative_Humidity': 'float64',
            'Approximate_Max': 'float64',
            'Solar_Radiation': 'float64',
            'Snow_cover': 'object',
            'Turfgrass_species': 'object',
            'Mowing_height': 'object',
            'Location': 'object',
            'Unnamed:_20': 'float64'
        })
        # Append the correctly typed row to the main DataFrame
        df=df.append(temp_df, ignore_index=True)
        Call_log( "Edge Layer", "CleaningSensorData", "Sensor Data has been Cleaned")
        print(df)
        
        
# # Run the server on localhost at port 12345
# udp_server('127.0.0.1', 12345)


# Main Function
Main function to start both CollectSensorData, AggregateSensorData, and listen_for_cloud_data .here UDP server is CollectSensorData,AggregateSensorData and listen_for_cloud_data use TCP protocol

In [ ]:

def main():
    # Get the IP address of the machine
    hostname = socket.gethostname()
    tcp_ip = socket.gethostbyname(hostname)

    # Set the ports
    udp_port = 12345
    tcp_send_port = 12346  # Port to send data to the cloud
    tcp_listen_port = 12347  # Port to receive data from the cloud
    # Physical server details
    #physical_ip = ##'192.168.137.59'##'127.0.0.1'  # IP of the physical server (assuming same machine)
    physical_ip='127.0.0.1'
    physical_port = 12348      # Port where the physical server is listening

    # Start UDP server in a separate thread
   # udp_thread = threading.Thread(target=CollectSensorData, args=('127.0.0.1', udp_port), daemon=True)
    udp_thread = threading.Thread(target=CollectSensorData, args=('0.0.0.0', udp_port), daemon=True)
    udp_thread.start()

    # Start the average calculation and sending in a separate thread
    threading.Thread(target=AggregateSensorData, args=(tcp_ip, tcp_send_port), daemon=True).start()

     # Start listening for data from the cloud
    listen_thread = threading.Thread(target=listen_for_cloud_data, args=(tcp_ip, tcp_listen_port,physical_ip, physical_port), daemon=True)
    listen_thread.start()

    # Keep the main thread alive
    udp_thread.join()
    listen_thread.join()

if __name__ == '__main__':
    main()

Server listening on 0.0.0.0:12345
print Df Edge server listening for cloud data on 192.168.0.20:12347
         DateTime
0  2/1/2022 16:30
Received data from ('127.0.0.1', 50061): ['2/23/2022 11:00', 32.9, 23.9, 11.066, 6.35589, -1.95762, 15.3454, 9.6412, 347.9, 1001.72, 29.5807, 0.0, 80.8768, 702.129, 38.0512, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
          DateTime  Soil_temperture_at_2_inch_below_the_soil_surface_(F)  \
0   2/1/2022 16:30                                                NaN      
1  2/23/2022 11:00                                               32.9      

   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  \
0                                                NaN            
1                                               23.9            

   Air_temperature_1.5m_(F)  Dewpoint  Wind_Chill  Wind_Gust_10m  \
0                       NaN       NaN         NaN            NaN   
1                    11.066   6.35589    -1.95762        15.3454   

   

C:\Users\Masrufa\AppData\Local\Temp\ipykernel_33728\3707936460.py:55: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df=df.append(temp_df, ignore_index=True)


          DateTime  Soil_temperture_at_2_inch_below_the_soil_surface_(F)  \
0   2/1/2022 16:30                                                NaN      
1  2/23/2022 11:00                                               32.9      

   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  \
0                                                NaN            
1                                               23.9            

   Air_temperature_1.5m_(F)  Dewpoint  Wind_Chill  Wind_Gust_10m  \
0                       NaN       NaN         NaN            NaN   
1                    11.066   6.35589    -1.95762        15.3454   

   Wind_Speed_10m  Wind_Direction_10m  Pressure_(mb)  Pressure_(inHg)  \
0             NaN                 NaN            NaN              NaN   
1          9.6412               347.9        1001.72          29.5807   

   Rainfall  Relative_Humidity  Approximate_Max  Solar_Radiation Snow_cover  \
0       NaN                NaN              NaN              NaN       

C:\Users\Masrufa\AppData\Local\Temp\ipykernel_33728\2994184367.py:13: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  grouped_df = df.groupby(['Snow_cover', 'Turfgrass_species', 'Mowing_height', 'Location']).mean().reset_index()


Connected to cloud server at ('192.168.0.20', 54653)
Data received from cloud: triggerActuator 1
Data forwarded to physical server: triggerActuator 1
Received data from ('127.0.0.1', 60809): ['2/6/2022 8:30', 32.9, 26.6, 25.9844, 20.8395, 22.1561, 3.87214, 3.23461, 115.5, 988.483, 29.1899, 0.19, 80.5671, 166.762, 48.3326, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
        DateTime  Soil_temperture_at_2_inch_below_the_soil_surface_(F)  \
0  2/6/2022 8:30                                               32.9      

   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  \
0                                               26.6            

   Air_temperature_1.5m_(F)  Dewpoint  Wind_Chill  Wind_Gust_10m  \
0                   25.9844   20.8395     22.1561        3.87214   

   Wind_Speed_10m  Wind_Direction_10m  Pressure_(mb)  Pressure_(inHg)  \
0         3.23461               115.5        988.483          29.1899   

   Rainfall  Relative_Humidity  Approximate_Max  Solar_R

...

C:\Users\Masrufa\AppData\Local\Temp\ipykernel_33728\3707936460.py:55: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df=df.append(temp_df, ignore_index=True)


Received data from ('127.0.0.1', 61008): ['2/26/2022 18:00', 39.2, 32.0, 40.8056, 19.9977, nan, 2.92368, 1.64191, 331.4, 995.353, 29.3928, 0.19, 42.8988, 37.6859, 30.1995, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
          DateTime  Soil_temperture_at_2_inch_below_the_soil_surface_(F)  \
0   2/26/2022 9:30                                               31.1      
1  2/26/2022 18:00                                               39.2      

   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  \
0                                               44.6            
1                                               32.0            

   Air_temperature_1.5m_(F)  Dewpoint Wind_Chill  Wind_Gust_10m  \
0                   25.2374   16.6048        NaN        3.58133   
1                   40.8056   19.9977        NaN        2.92368   

   Wind_Speed_10m  Wind_Direction_10m  Pressure_(mb)  Pressure_(inHg)  \
0         2.28391               69.02        999.892          29.5268   


C:\Users\Masrufa\AppData\Local\Temp\ipykernel_33728\3707936460.py:55: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df=df.append(temp_df, ignore_index=True)


          DateTime  Soil_temperture_at_2_inch_below_the_soil_surface_(F)  \
0   2/26/2022 9:30                                               31.1      
1  2/26/2022 18:00                                               39.2      

   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  \
0                                               44.6            
1                                               32.0            

   Air_temperature_1.5m_(F)  Dewpoint Wind_Chill  Wind_Gust_10m  \
0                   25.2374   16.6048        NaN        3.58133   
1                   40.8056   19.9977        NaN        2.92368   

   Wind_Speed_10m  Wind_Direction_10m  Pressure_(mb)  Pressure_(inHg)  \
0         2.28391               69.02        999.892          29.5268   
1         1.64191              331.40        995.353          29.3928   

   Rainfall  Relative_Humidity  Approximate_Max  Solar_Radiation Snow_cover  \
0      0.00            69.2898         475.3010         323.1130         N

C:\Users\Masrufa\AppData\Local\Temp\ipykernel_33728\2994184367.py:13: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  grouped_df = df.groupby(['Snow_cover', 'Turfgrass_species', 'Mowing_height', 'Location']).mean().reset_index()


In [ ]:
df